# Notebook 02: Preprocessing

**Purpose:** One-hot encode UGRansome2024 features and perform a stratified 80/20 split. Sample and clean CICIoT2023 using the official pre-split files, then binarize labels.

**Outputs:** `data/processed/ugr_train.csv`, `ugr_test.csv`, `cic_train.csv`, `cic_test.csv`

In [1]:
import os
# Use the repository root as the working directory, whether this notebook is
# launched from the repo root or from the notebooks/ folder.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Libraries loaded.')

Libraries loaded.


## Part 1: UGRansome2024

In [2]:
ugr = pd.read_csv('data/processed/ugr_clean.csv')
print(f'Loaded ugr_clean.csv: {ugr.shape}')
print(f'Columns: {ugr.columns.tolist()}')

Loaded ugr_clean.csv: (89859, 11)
Columns: ['Time', 'Protocol', 'Flag', 'Family', 'Clusters', 'USD', 'Netflow_Bytes', 'IPaddress', 'Threats', 'Port', 'Prediction']


In [3]:
# One-hot encode categorical columns
CAT_COLS = ['Protocol', 'Flag', 'IPaddress', 'Family', 'Threats', 'Clusters']
ugr_encoded = pd.get_dummies(ugr, columns=CAT_COLS, drop_first=False)
print(f'Shape after one-hot encoding: {ugr_encoded.shape}')
print(f'New columns: {ugr_encoded.shape[1] - ugr.shape[1] + len(CAT_COLS)} added')

Shape after one-hot encoding: (89859, 50)
New columns: 45 added


In [4]:
# Stratified 80/20 split
X = ugr_encoded.drop(columns=['Prediction'])
y = ugr_encoded['Prediction']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

ugr_train = pd.concat([X_train, y_train], axis=1)
ugr_test  = pd.concat([X_test,  y_test],  axis=1)

print(f'UGR train shape: {ugr_train.shape}, attack rate: {ugr_train["Prediction"].mean():.4f}')
print(f'UGR test  shape: {ugr_test.shape},  attack rate: {ugr_test["Prediction"].mean():.4f}')

UGR train shape: (71887, 50), attack rate: 0.2279
UGR test  shape: (17972, 50),  attack rate: 0.2280


In [5]:
ugr_train.to_csv('data/processed/ugr_train.csv', index=False)
ugr_test.to_csv('data/processed/ugr_test.csv',   index=False)
print('UGR splits saved.')

UGR splits saved.


## Part 2: CICIoT2023

In [6]:
# Stratified sample from train.csv (target 160k rows)
TRAIN_TARGET = 160_000
TEST_TARGET  = 40_000

train_path = 'data/raw/CICIOT23/train/train.csv'
test_path  = 'data/raw/CICIOT23/test/test.csv'

# Count rows to compute fraction
train_total = 5_491_971
test_total  = 1_176_851
train_frac  = TRAIN_TARGET / train_total
test_frac   = TEST_TARGET  / test_total
print(f'Train sample fraction: {train_frac:.5f}')
print(f'Test  sample fraction: {test_frac:.5f}')

Train sample fraction: 0.02913
Test  sample fraction: 0.03399


In [7]:
# Load train sample in chunks to avoid memory overflow
print('Loading CICIoT train sample ...')
chunks = []
for chunk in pd.read_csv(train_path, chunksize=200_000, low_memory=False):
    chunks.append(chunk.sample(frac=train_frac, random_state=RANDOM_STATE))
cic_train_raw = pd.concat(chunks, ignore_index=True)
del chunks
print(f'CICIoT train sample loaded: {cic_train_raw.shape}')

Loading CICIoT train sample ...


CICIoT train sample loaded: (160008, 47)


In [8]:
# Load test sample in chunks
print('Loading CICIoT test sample ...')
chunks = []
for chunk in pd.read_csv(test_path, chunksize=200_000, low_memory=False):
    chunks.append(chunk.sample(frac=test_frac, random_state=RANDOM_STATE))
cic_test_raw = pd.concat(chunks, ignore_index=True)
del chunks
print(f'CICIoT test sample loaded: {cic_test_raw.shape}')

Loading CICIoT test sample ...


CICIoT test sample loaded: (40001, 47)


In [9]:
def clean_cic(df, name):
    before = len(df)
    # Replace inf with NaN
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
    # Drop NaN rows
    df = df.dropna()
    # Remove duplicates
    df = df.drop_duplicates()
    print(f'{name}: {before} -> {len(df)} rows after cleaning')
    return df.reset_index(drop=True)

cic_train_raw = clean_cic(cic_train_raw, 'cic_train')
cic_test_raw  = clean_cic(cic_test_raw,  'cic_test')

cic_train: 160008 -> 159874 rows after cleaning
cic_test: 40001 -> 39995 rows after cleaning


In [10]:
# Binarize: BenignTraffic -> 0, all attacks -> 1
# Keep original multi-class 'label' column for Notebook 12
cic_train_raw['label_binary'] = (cic_train_raw['label'] != 'BenignTraffic').astype(int)
cic_test_raw['label_binary']  = (cic_test_raw['label']  != 'BenignTraffic').astype(int)

print('CICIoT train label distribution:')
print(cic_train_raw['label_binary'].value_counts())
print(f'Attack rate: {cic_train_raw["label_binary"].mean():.4f}')

print('\nCICIoT test label distribution:')
print(cic_test_raw['label_binary'].value_counts())
print(f'Attack rate: {cic_test_raw["label_binary"].mean():.4f}')

CICIoT train label distribution:
label_binary
1    156139
0      3735
Name: count, dtype: int64
Attack rate: 0.9766

CICIoT test label distribution:
label_binary
1    39049
0      946
Name: count, dtype: int64
Attack rate: 0.9763


In [11]:
# Confirm multi-class label column preserved
print('Unique attack categories in cic_train:')
print(cic_train_raw['label'].value_counts())

Unique attack categories in cic_train:
label
DDoS-ICMP_Flood            24728
DDoS-UDP_Flood             18707
DDoS-TCP_Flood             15260
DDoS-SYN_Flood             13952
DDoS-PSHACK_Flood          13895
DDoS-RSTFINFlood           13790
DDoS-SynonymousIP_Flood    12312
DoS-UDP_Flood              11552
DoS-TCP_Flood               9227
DoS-SYN_Flood               6868
BenignTraffic               3735
Mirai-greeth_flood          3345
Mirai-udpplain              3028
Mirai-greip_flood           2583
DDoS-ICMP_Fragmentation     1467
MITM-ArpSpoofing            1069
DDoS-UDP_Fragmentation      1010
DDoS-ACK_Fragmentation       992
DNS_Spoofing                 633
Recon-HostDiscovery          441
Recon-OSScan                 340
Recon-PortScan               303
DoS-HTTP_Flood               218
VulnerabilityScan            122
DDoS-HTTP_Flood               85
DDoS-SlowLoris                79
DictionaryBruteForce          55
BrowserHijacking              21
CommandInjection              1

In [12]:
cic_train_raw.to_csv('data/processed/cic_train.csv', index=False)
cic_test_raw.to_csv('data/processed/cic_test.csv',   index=False)
print('CICIoT splits saved.')
print(f'cic_train: {cic_train_raw.shape}')
print(f'cic_test:  {cic_test_raw.shape}')

CICIoT splits saved.
cic_train: (159874, 48)
cic_test:  (39995, 48)


In [13]:
# Summary of feature counts and class distributions
ugr_t = pd.read_csv('data/processed/ugr_train.csv', nrows=1)
cic_t = pd.read_csv('data/processed/cic_train.csv', nrows=1)
print('=== Feature summary ===')
print(f'UGR features (excl. target): {ugr_t.shape[1] - 1}')
print(f'CIC features (excl. label cols): {cic_t.shape[1] - 2}')

print('\n=== Row counts ===')
print(f'UGR train: {len(pd.read_csv("data/processed/ugr_train.csv"))}')
print(f'UGR test:  {len(pd.read_csv("data/processed/ugr_test.csv"))}')
print(f'CIC train: {len(cic_train_raw)}')
print(f'CIC test:  {len(cic_test_raw)}')

=== Feature summary ===
UGR features (excl. target): 49
CIC features (excl. label cols): 46

=== Row counts ===
UGR train: 71887
UGR test:  17972
CIC train: 159874
CIC test:  39995


## Summary

**Purpose:** Prepare train/test splits for both datasets.

**Method (UGRansome):** One-hot encoded six categorical columns (Protocol, Flag, IPaddress, Family, Threats, Clusters). Applied stratified 80/20 split with random_state=42.

**Method (CICIoT2023):** Loaded stratified samples from the official pre-split train and test files (160k train / 40k test) to stay within VPS memory limits. Replaced infinite values, dropped NaN rows and duplicates. Binarized labels: BenignTraffic=0, all attack types=1. Preserved the original multi-class label column for per-category analysis in Notebook 12.

**Key findings:** See printed output for final feature counts and class distributions.